### Setup toy data

In [1]:
import numpy as np
from numpy.typing import NDArray
from scipy.stats import norm
from tqdm import tqdm
import matplotlib.pyplot as plt

FloatArray = NDArray[np.float64]


def generate_toy_data(
    num_samples: int,
    feature_dim: int = 256,
    num_ground_truth_features: int = 512,
    num_active_features: float = 5.0,
    decay_rate: float = 0.99,
    random_seed: int | None = None,
) -> tuple[FloatArray, FloatArray, FloatArray, FloatArray]:
    """
    Generate toy data for sparse feature-learning experiments.

    Generation steps:
    1. Sample `num_ground_truth_features` random directions in R^{feature_dim}
       and normalize each one to unit norm.
    2. Build a random positive-semidefinite covariance matrix across latent
       features.
    3. Sample correlated Gaussian latents for each synthetic sample.
    4. Map those Gaussian latents through the standard normal CDF to get
       correlated values in (0, 1).
    5. Apply index-based decay so lower-index features are generally more
       common.
    6. Rescale probabilities so the expected number of active features is
       `num_active_features`.
    7. Draw Bernoulli support masks and assign Uniform(0, 1) amplitudes to
       active features.
    8. Form each observed sample as a linear combination of normalized ground-
       truth feature directions.

    Returns
    -------
    dataset : NDArray[np.float64], shape (num_samples, feature_dim)
    ground_truth_features : NDArray[np.float64], shape (feature_dim, num_ground_truth_features)
    sparse_coefficients : NDArray[np.float64], shape (num_samples, num_ground_truth_features)
    final_probabilities : NDArray[np.float64], shape (num_samples, num_ground_truth_features)
    """

    # Use assert statements for basic input validation, with informative error messages.
    assert num_samples > 0, f"num_samples must be positive, got {num_samples}."
    assert feature_dim > 0, f"feature_dim must be positive, got {feature_dim}."
    assert (
        num_ground_truth_features > 0
    ), f"num_ground_truth_features must be positive, got {num_ground_truth_features}."
    assert (
        num_active_features >= 0
    ), f"num_active_features must be nonnegative, got {num_active_features}."
    assert decay_rate > 0, f"decay_rate must be positive, got {decay_rate}."

    rng = np.random.default_rng(random_seed)

    # ------------------------------------------------------------------
    # 1) Sample ground-truth feature directions, then normalize columns.
    #    Shape: (feature_dim, num_ground_truth_features)
    # ------------------------------------------------------------------
    ground_truth_features: FloatArray = rng.standard_normal(
        size=(feature_dim, num_ground_truth_features)
    )
    column_norms: FloatArray = np.linalg.norm(ground_truth_features, axis=0)
    column_norms = np.where(column_norms == 0.0, 1.0, column_norms)
    ground_truth_features = ground_truth_features / column_norms

    # ------------------------------------------------------------------
    # 2) Build a random covariance matrix for correlated latent features.
    # ------------------------------------------------------------------
    covariance_seed: FloatArray = rng.standard_normal(
        size=(num_ground_truth_features, num_ground_truth_features)
    )
    covariance: FloatArray = covariance_seed @ covariance_seed.T

    # ------------------------------------------------------------------
    # 3) Draw correlated Gaussian latents for all samples at once.
    # ------------------------------------------------------------------
    mean: FloatArray = np.zeros(num_ground_truth_features, dtype=np.float64)
    gaussian_samples: FloatArray = rng.multivariate_normal(
        mean,
        covariance,
        size=num_samples,
    )

    # ------------------------------------------------------------------
    # 4) Map Gaussian samples through the standard normal CDF.
    # ------------------------------------------------------------------
    correlated_feature_probs: FloatArray = norm.cdf(gaussian_samples)

    # ------------------------------------------------------------------
    # 5) Allocate outputs and precompute feature indices for decay.
    # ------------------------------------------------------------------
    sparse_coefficients: FloatArray = np.zeros(
        (num_samples, num_ground_truth_features), dtype=np.float64
    )
    dataset: FloatArray = np.zeros((num_samples, feature_dim), dtype=np.float64)
    
    feature_indices: FloatArray = np.arange(num_ground_truth_features, dtype=np.float64)

    # ------------------------------------------------------------------
    # 6) Per sample, apply decay and rescale probabilities to match the
    #    desired expected active-feature count.
    # 7) Sample Bernoulli support and Uniform(0,1) amplitudes.
    # 8) Form observed vector by linear combination of latent features.
    # ------------------------------------------------------------------
    for i in tqdm(range(num_samples)):
        decayed_feature_probs: FloatArray = np.power(
            correlated_feature_probs[i],
            feature_indices * decay_rate,
        )

        mean_prob = float(np.mean(decayed_feature_probs))
        if mean_prob <= 0.0:
            raise ValueError(
                "Mean decayed probability became non-positive, which should "
                "not happen with valid inputs."
            )

        rescaled_probs: FloatArray = decayed_feature_probs / mean_prob
        rescaled_probs = (
            num_active_features * rescaled_probs / num_ground_truth_features
        )

        binary_vector: FloatArray = rng.binomial(1, rescaled_probs).astype(np.float64)
        activations: FloatArray = binary_vector * rng.uniform(
            0.0,
            1.0,
            num_ground_truth_features,
        )

        sparse_coefficients[i] = activations
        dataset[i] = ground_truth_features @ activations

    return dataset, ground_truth_features, sparse_coefficients

In [3]:
# Setup deterministic seed
SEED_NUM = 42

def setup_seed(seed: int) -> None:
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

setup_seed(SEED_NUM)

In [4]:
# Generate the toy dataset with N samples
num_samples = 100000
feature_dim = 256
num_ground_truth_features = 512
num_active_features = 5
dataset, ground_truth_features, sparse_coefficients = generate_toy_data(
    num_samples=num_samples,
    feature_dim=feature_dim,
    num_ground_truth_features=num_ground_truth_features,
    num_active_features=num_active_features,
    decay_rate=0.99,
    random_seed=SEED_NUM,)

100%|██████████| 100000/100000 [00:03<00:00, 27274.36it/s]
